# 02 — Text Preprocessing Pipeline

This notebook transforms the raw IMDb CSV into the encoded sequences and vocabulary that the model notebooks will consume.
It follows a strict pipeline:

1. **Load and Split** — divide the dataset into train / val / test before touching the text
2. **Clean Text** — normalise raw reviews (lowercase, strip HTML, remove punctuation)
3. **Build Vocabulary** — learn the token-to-index mapping from training data only
4. **Encode Sequences** — convert every review to a fixed-length integer array
5. **Verify Round-Trip** — confirm that encoding then decoding recovers the original cleaned tokens

Each section is preceded by a rationale cell and followed by an interpretation cell so the notebook tells a coherent story.

## Setup

Add the repository root to `sys.path` so that `src/preprocessing.py` is importable both locally and in Google Colab.

In [1]:
import sys, os

# ── Colab vs. local ortam tespiti ────────────────────────────────────────
if os.path.exists("/content"):
    repo_root = "/content/IMDb_Sentiment_Analysis"
    if not os.path.exists(repo_root):
        os.system("git clone https://github.com/azrakarakaya1/IMDb_Sentiment_Analysis.git " + repo_root)
    os.chdir(repo_root)
else:
    os.chdir(os.path.abspath(".."))

sys.path.insert(0, os.path.abspath("."))
print(f"Working directory: {os.getcwd()}")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

Working directory: /Users/azrakarakaya/IMDb_Sentiment_Analysis


## Section 1 — Load and Split

### Why split before building the vocabulary?

A fundamental rule in machine learning is that **no information from the validation or test sets should influence any preprocessing decision**.
If we built the vocabulary from all 50 000 reviews, the model would indirectly "see" words that appear only in the test set,
giving it an unfair advantage and producing optimistically biased evaluation metrics — a form of **data leakage**.

By splitting first, we guarantee that the vocabulary, and every downstream statistic derived from it, is computed solely from training data.

### Why 80 / 10 / 10 with seed = 42?

- **80 % train** gives the model 40 000 examples — enough to learn a rich vocabulary and generalise well.
- **10 % validation** (5 000 examples) provides a stable signal for early stopping and hyperparameter tuning without wasting too much data.
- **10 % test** (5 000 examples) is held out entirely until final evaluation, giving an unbiased estimate of real-world performance.
- **`random_state=42`** makes the split deterministic so that every run of this notebook produces identical splits, ensuring reproducibility.


In [2]:
# Load the raw dataset
DATA_PATH = "data/IMDB_Dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH!r}.\n"
        "Please place IMDB_Dataset.csv in the data/ directory before running this notebook."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows with columns: {list(df.columns)}")

# ── First split: carve off 20 % as a temporary val+test pool ──────────────
# test_size=0.2 means 80 % goes to train, 20 % goes to temp
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# ── Second split: divide the 20 % pool equally into val and test ──────────
# test_size=0.5 means 50 % of temp → val, 50 % of temp → test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# ── Sanity-check the sizes ────────────────────────────────────────────────
assert len(train_df) == 40_000, f"Expected 40000 train rows, got {len(train_df)}"
assert len(val_df)   ==  5_000, f"Expected 5000 val rows, got {len(val_df)}"
assert len(test_df)  ==  5_000, f"Expected 5000 test rows, got {len(test_df)}"

# ── Display a summary table ───────────────────────────────────────────────
summary = pd.DataFrame({
    "Split":    ["Train", "Validation", "Test", "Total"],
    "Rows":     [len(train_df), len(val_df), len(test_df), len(df)],
    "Fraction": ["80 %", "10 %", "10 %", "100 %"],
})
print(summary.to_string(index=False))


Loaded 50,000 rows with columns: ['review', 'sentiment']
     Split  Rows Fraction
     Train 40000     80 %
Validation  5000     10 %
      Test  5000     10 %
     Total 50000    100 %


**Interpretation:** The three splits sum to 50 000 and match the expected 80 / 10 / 10 ratio.
The `assert` statements above will raise an error immediately if the split sizes are wrong,
making this notebook fail fast rather than silently producing incorrect downstream artefacts.


## Section 2 — Clean Text

`clean_text` applies three normalisation steps in order:

1. **Lowercase** — `"Movie"` and `"movie"` should be the same token.
   Without this step the vocabulary would contain duplicate entries that differ only in capitalisation.

2. **Strip HTML tags** — IMDb reviews frequently contain HTML markup such as `<br />`, `<b>`, and `</em>`.
   These tags carry no sentiment signal and would pollute the vocabulary with meaningless tokens.
   The regex `<[^>]+>` matches any tag (opening or closing) and replaces it with a space
   so that adjacent words are not accidentally merged (e.g. `"great</b>film"` → `"great film"`).

3. **Remove non-alphabetic characters** — punctuation, digits, and special characters are removed
   with `[^a-z\s]`.  This keeps the vocabulary clean and avoids the model learning spurious
   associations between punctuation patterns and sentiment.


In [3]:
from src.preprocessing import clean_text

# Apply clean_text to every review in all three splits
# .copy() prevents SettingWithCopyWarning since train_df is a slice of df
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df["clean"] = train_df["review"].apply(clean_text)
val_df["clean"]   = val_df["review"].apply(clean_text)
test_df["clean"]  = test_df["review"].apply(clean_text)

print("Cleaning complete.")
print(f"  Train rows: {len(train_df):,}")
print(f"  Val rows:   {len(val_df):,}")
print(f"  Test rows:  {len(test_df):,}")

# ── Before / after comparison for 3 representative reviews ───────────────
# Pick one review with HTML, one with heavy punctuation, one normal
examples = [
    train_df.iloc[0],
    train_df.iloc[1],
    train_df.iloc[2],
]

for i, row in enumerate(examples, 1):
    raw     = row["review"]
    cleaned = row["clean"]
    print(f"\n--- Example {i} (sentiment: {row['sentiment']}) ---")
    print(f"  BEFORE ({len(raw)} chars):   {raw[:120]!r}")
    print(f"  AFTER  ({len(cleaned)} chars): {cleaned[:120]!r}")


Cleaning complete.
  Train rows: 40,000
  Val rows:   5,000
  Test rows:  5,000

--- Example 1 (sentiment: negative) ---
  BEFORE (2328 chars):   "That's what I kept asking myself during the many fights, screaming matches, swearing and general mayhem that permeate th"
  AFTER  (2222 chars): 'thats what i kept asking myself during the many fights screaming matches swearing and general mayhem that permeate the  '

--- Example 2 (sentiment: negative) ---
  BEFORE (1624 chars):   'I did not watch the entire movie. I could not watch the entire movie. I stopped the DVD after watching for half an hour '
  AFTER  (1561 chars): 'i did not watch the entire movie i could not watch the entire movie i stopped the dvd after watching for half an hour an'

--- Example 3 (sentiment: positive) ---
  BEFORE (502 chars):   "A touching love story reminiscent of \x91In the Mood for Love'. Drawing heavily on Chinese poetry and how this is used by e"
  AFTER  (492 chars): 'a touching love story reminiscent of

**Interpretation:** The before/after comparison shows that HTML tags have been removed,
all text is lowercase, and punctuation is gone.
The cleaned text may be longer in character count than the raw text in some cases because
HTML tags are replaced with spaces rather than deleted outright — this prevents word merging.
The cleaned strings are now ready for tokenisation.


## Section 3 — Build Vocabulary

### Why build the vocabulary from training data only?

Building the vocabulary from all 50 000 reviews would be another form of data leakage:
the model would know which words appear in the test set, even if it never sees their labels.
By restricting vocabulary construction to the 40 000 training reviews, we ensure that
any word that appears only in validation or test data is treated as `<UNK>` — exactly
as it would be in a real deployment where the model encounters unseen vocabulary.

### Why cap at 20 000 tokens?

The IMDb corpus contains tens of thousands of unique tokens, but the vast majority appear
only once or twice and carry little predictive signal.  Keeping the top 20 000 tokens
by frequency covers approximately 95 % of the token mass while keeping the embedding
table small (20 000 × 128 = 2.56 M parameters — manageable on a free Colab GPU).

### Special indices

- **`PAD_IDX = 0`** — the padding token.  Reviews shorter than 500 tokens are padded
  with zeros at the end.  The embedding layer uses `padding_idx=0` so that padding
  positions contribute zero gradients during training.
- **`UNK_IDX = 1`** — the unknown token.  Any word not in the top 20 000 is mapped here.
  This is a deliberate design choice: rather than crashing on unseen words, the model
  learns a generic "unknown word" representation.


In [4]:
from src.preprocessing import Vocabulary

# Tokenise the cleaned training reviews into lists of tokens
# str.split() splits on any whitespace and discards empty strings
train_token_lists = [text.split() for text in train_df["clean"]]

# Build the vocabulary from training tokens ONLY
vocab = Vocabulary(max_size=20_000)
vocab.build(train_token_lists)

print(f"Vocabulary size (including <PAD> and <UNK>): {len(vocab):,}")
print(f"  PAD token: {vocab.PAD_TOKEN!r} → index {vocab.PAD_IDX}")
print(f"  UNK token: {vocab.UNK_TOKEN!r} → index {vocab.UNK_IDX}")

# ── Most common 20 tokens ─────────────────────────────────────────────────
# The internal _token_to_idx dict is ordered by insertion (Python 3.7+),
# and tokens were inserted in descending frequency order starting at index 2.
most_common_tokens = [
    (token, idx)
    for token, idx in vocab._token_to_idx.items()
    if idx >= 2  # skip <PAD> and <UNK>
][:20]

print("\nMost common 20 tokens:")
for token, idx in most_common_tokens:
    print(f"  idx {idx:5d}: {token}")

# ── Least common 20 tokens ────────────────────────────────────────────────
least_common_tokens = [
    (token, idx)
    for token, idx in vocab._token_to_idx.items()
    if idx >= 2
][-20:]

print("\nLeast common 20 tokens (tail of vocabulary):")
for token, idx in least_common_tokens:
    print(f"  idx {idx:5d}: {token}")

# ── Save vocabulary to disk ───────────────────────────────────────────────
os.makedirs("data", exist_ok=True)  # ensure data/ directory exists
vocab.save("data/vocab.pkl")
print("\nVocabulary saved to data/vocab.pkl")


Vocabulary size (including <PAD> and <UNK>): 20,002
  PAD token: '<PAD>' → index 0
  UNK token: '<UNK>' → index 1

Most common 20 tokens:
  idx     2: the
  idx     3: and
  idx     4: a
  idx     5: of
  idx     6: to
  idx     7: is
  idx     8: in
  idx     9: it
  idx    10: i
  idx    11: this
  idx    12: that
  idx    13: was
  idx    14: as
  idx    15: for
  idx    16: with
  idx    17: movie
  idx    18: but
  idx    19: film
  idx    20: on
  idx    21: not

Least common 20 tokens (tail of vocabulary):
  idx 19982: dragnet
  idx 19983: bathe
  idx 19984: frothy
  idx 19985: uproar
  idx 19986: alumni
  idx 19987: denizens
  idx 19988: preppy
  idx 19989: upfront
  idx 19990: capped
  idx 19991: unjustified
  idx 19992: underscore
  idx 19993: bingo
  idx 19994: amateurishly
  idx 19995: damatos
  idx 19996: beguiling
  idx 19997: respecting
  idx 19998: freeing
  idx 19999: tonk
  idx 20000: bailed
  idx 20001: insignificance

Vocabulary saved to data/vocab.pkl


**Interpretation:** The most common tokens are function words like "the", "a", "and" — this is expected
for any English corpus.  The least common tokens in the vocabulary are rare but still appeared
frequently enough to make the top 20 000 cut.  Any token beyond rank 20 000 will be mapped to `<UNK>`
at encode time.  The vocabulary has been serialised to `data/vocab.pkl` so that later notebooks
can reload it without rebuilding from scratch.


## Section 4 — Encode Sequences

### The encoding process

`Tokenizer.encode(text)` applies the full pipeline in one call:

1. **Tokenise** — `clean_text(text).split()` produces a list of lowercase alphabetic tokens.
2. **Vocabulary lookup** — each token is mapped to its integer index; unknown tokens become `UNK_IDX = 1`.
3. **Truncate** — if the review has more than 500 tokens, only the first 500 are kept.
   We keep the *beginning* of the review because IMDb reviews typically state their overall
   sentiment early ("This film was terrible...").
4. **Post-pad** — if the review has fewer than 500 tokens, zeros (`PAD_IDX`) are appended
   at the *end* until the sequence reaches length 500.

### What padding looks like

A short review of 50 tokens produces a sequence like:
`[42, 7, 315, ..., 0, 0, 0, 0, 0]` — 50 real indices followed by 450 zeros.

### What truncation looks like

A long review of 800 tokens produces a sequence like:
`[42, 7, 315, ..., 1024]` — exactly 500 indices, the last 300 tokens discarded.

### Label encoding

`"positive"` → `1.0` and `"negative"` → `0.0` as `float32` arrays.
Float32 is required because PyTorch's `BCELoss` expects floating-point targets.


In [5]:
from src.preprocessing import Tokenizer

# Create a tokenizer backed by the vocabulary we just built
tokenizer = Tokenizer(vocab=vocab, max_len=500)

# ── Encode reviews ────────────────────────────────────────────────────────
# Each call to tokenizer.encode returns a list of 500 ints;
# np.array(..., dtype=np.int32) stacks them into a 2-D array of shape (N, 500)
print("Encoding train sequences...")
train_sequences = np.array(
    [tokenizer.encode(text) for text in train_df["review"]], dtype=np.int32
)

print("Encoding val sequences...")
val_sequences = np.array(
    [tokenizer.encode(text) for text in val_df["review"]], dtype=np.int32
)

print("Encoding test sequences...")
test_sequences = np.array(
    [tokenizer.encode(text) for text in test_df["review"]], dtype=np.int32
)

# ── Encode labels ─────────────────────────────────────────────────────────
# Map "positive" → 1.0 and "negative" → 0.0; float32 for BCELoss compatibility
label_map = {"positive": 1.0, "negative": 0.0}

train_labels = np.array(
    [label_map[s] for s in train_df["sentiment"]], dtype=np.float32
)
val_labels = np.array(
    [label_map[s] for s in val_df["sentiment"]], dtype=np.float32
)
test_labels = np.array(
    [label_map[s] for s in test_df["sentiment"]], dtype=np.float32
)

# ── Save all arrays to disk ───────────────────────────────────────────────
np.save("data/train_sequences.npy", train_sequences)
np.save("data/train_labels.npy",    train_labels)
np.save("data/val_sequences.npy",   val_sequences)
np.save("data/val_labels.npy",      val_labels)
np.save("data/test_sequences.npy",  test_sequences)
np.save("data/test_labels.npy",     test_labels)

# ── Print shapes of all saved arrays ─────────────────────────────────────
print("\nSaved array shapes:")
print(f"  train_sequences : {train_sequences.shape}  dtype={train_sequences.dtype}")
print(f"  train_labels    : {train_labels.shape}  dtype={train_labels.dtype}")
print(f"  val_sequences   : {val_sequences.shape}  dtype={val_sequences.dtype}")
print(f"  val_labels      : {val_labels.shape}  dtype={val_labels.dtype}")
print(f"  test_sequences  : {test_sequences.shape}  dtype={test_sequences.dtype}")
print(f"  test_labels     : {test_labels.shape}  dtype={test_labels.dtype}")


Encoding train sequences...
Encoding val sequences...
Encoding test sequences...

Saved array shapes:
  train_sequences : (40000, 500)  dtype=int32
  train_labels    : (40000,)  dtype=float32
  val_sequences   : (5000, 500)  dtype=int32
  val_labels      : (5000,)  dtype=float32
  test_sequences  : (5000, 500)  dtype=int32
  test_labels     : (5000,)  dtype=float32


**Interpretation:** Every sequence array has shape `(N, 500)` with dtype `int32`,
and every label array has shape `(N,)` with dtype `float32`.
The second dimension of 500 is fixed regardless of the original review length —
short reviews are padded and long reviews are truncated.
These arrays are now ready to be wrapped in a `torch.utils.data.Dataset` in the training notebook.


## Section 5 — Verify Round-Trip

### What the round-trip property guarantees

The round-trip property states:

> For any input text, `tokenizer.decode(tokenizer.encode(text))` equals `tokenizer.tokenize(text)[:500]`
> (after applying the same vocabulary mapping to both sides).

This matters for two reasons:

1. **Correctness** — it confirms that encoding and decoding are consistent inverses of each other
   (up to truncation and UNK substitution).  If this property failed, it would indicate a bug
   in the vocabulary mapping or the padding/truncation logic.

2. **Debuggability** — when a model makes a surprising prediction, we can decode the input
   sequence back to tokens and inspect exactly what the model "saw".  The round-trip guarantee
   means this decoded view is faithful to the actual model input.

**Note on UNK tokens:** If a token is not in the vocabulary, it is encoded as `UNK_IDX = 1`
and decoded back as `"<UNK>"`.  The assertion therefore compares the decoded output to the
tokenized-then-vocab-mapped output (both go through the same vocabulary lookup path).


In [6]:
# ── Demonstrate round-trip for 3 training examples ───────────────────────
for i in range(3):
    raw_text     = train_df["review"].iloc[i]          # original raw review
    cleaned_text = train_df["clean"].iloc[i]           # after clean_text
    encoded      = tokenizer.encode(raw_text)           # list of 500 ints
    decoded      = tokenizer.decode(encoded)            # list of tokens (PAD stripped)

    print(f"\n=== Example {i+1} ===")
    print(f"1. Original raw text (first 120 chars):")
    print(f"   {raw_text[:120]!r}")
    print(f"2. Cleaned text (first 120 chars):")
    print(f"   {cleaned_text[:120]!r}")
    print(f"3. Encoded sequence (first 20 indices):")
    print(f"   {encoded[:20]}")
    print(f"4. Decoded sequence (first 20 tokens):")
    print(f"   {decoded[:20]}")

# ── Assert round-trip property for the same 3 examples ───────────────────
print("\nVerifying round-trip property...")
for i in range(3):
    raw_text = train_df["review"].iloc[i]

    # Encode then decode (padding stripped by tokenizer.decode)
    decoded = tokenizer.decode(tokenizer.encode(raw_text))

    # The expected result: tokenize the text and take the first 500 tokens
    # tokenizer.tokenize calls clean_text then str.split()
    expected = tokenizer.tokenize(raw_text)[:500]

    # Both sides go through the same vocabulary, so UNK substitution is symmetric
    # Re-encode expected tokens through vocab to apply the same UNK mapping
    expected_via_vocab = vocab.decode(vocab.encode(expected))

    assert decoded == expected_via_vocab, (
        f"Round-trip failed for example {i+1}:\n"
        f"  decoded:  {decoded[:10]}\n"
        f"  expected: {expected_via_vocab[:10]}"
    )
    print(f"  Example {i+1}: round-trip OK ({len(decoded)} tokens)")

print("\nAll round-trip assertions passed.")



=== Example 1 ===
1. Original raw text (first 120 chars):
   "That's what I kept asking myself during the many fights, screaming matches, swearing and general mayhem that permeate th"
2. Cleaned text (first 120 chars):
   'thats what i kept asking myself during the many fights screaming matches swearing and general mayhem that permeate the  '
3. Encoded sequence (first 20 indices):
   [176, 48, 10, 770, 2064, 514, 292, 2, 105, 1882, 1957, 4180, 6328, 3, 793, 4796, 12, 1, 2, 225]
4. Decoded sequence (first 20 tokens):
   ['thats', 'what', 'i', 'kept', 'asking', 'myself', 'during', 'the', 'many', 'fights', 'screaming', 'matches', 'swearing', 'and', 'general', 'mayhem', 'that', '<UNK>', 'the', 'minutes']

=== Example 2 ===
1. Original raw text (first 120 chars):
   'I did not watch the entire movie. I could not watch the entire movie. I stopped the DVD after watching for half an hour '
2. Cleaned text (first 120 chars):
   'i did not watch the entire movie i could not watch the entire mo

**Interpretation:** The round-trip assertions confirm that the encoding pipeline is internally consistent.
For each example, the decoded token list matches the tokenized-then-vocab-mapped token list exactly.
Any token that was mapped to `<UNK>` during encoding is decoded back as `"<UNK>"` — this is expected
and correct behaviour, not a bug.

### Summary of artefacts produced by this notebook

| File | Shape / Size | Description |
|------|-------------|-------------|
| `data/vocab.pkl` | ≤ 20 002 entries | Token-to-index mapping |
| `data/train_sequences.npy` | (40000, 500) int32 | Encoded training reviews |
| `data/train_labels.npy` | (40000,) float32 | Training labels (0 or 1) |
| `data/val_sequences.npy` | (5000, 500) int32 | Encoded validation reviews |
| `data/val_labels.npy` | (5000,) float32 | Validation labels |
| `data/test_sequences.npy` | (5000, 500) int32 | Encoded test reviews |
| `data/test_labels.npy` | (5000,) float32 | Test labels |

These files are consumed by `03_model_building.ipynb` and `04_training.ipynb`.


In [ ]:
# ── Dosyaları Drive'a kaydet (session kapanınca kaybolmasın) ──────────────
if os.path.exists('/content/drive'):
    import shutil
    drive_dir = '/content/drive/MyDrive/IMDb_preprocessed'
    os.makedirs(drive_dir, exist_ok=True)
    for fname in ['vocab.pkl', 'train_sequences.npy', 'train_labels.npy',
                   'val_sequences.npy', 'val_labels.npy',
                   'test_sequences.npy', 'test_labels.npy']:
        shutil.copy(f'data/{fname}', f'{drive_dir}/{fname}')
    print(f'Tüm dosyalar Drive kaydedildi: {drive_dir}')
else:
    print('Drive bağlı değil, yerel kayıt yeterli.')